In [4]:
import numpy as  np
import pandas as pd

# Import functions for interpolation
from scipy.interpolate import RegularGridInterpolator 
from scipy.optimize import minimize

In [5]:
def read_star_row_from_csv(
    host_name: str,
    sigma_mag: bool = True,
    sigma_parallax: float = 0.1,
):
    master_csv_path = "ASTR502_Mega_Target_List_1.csv"
    master_phot_csv_path = "ASTR502_Master_Photometry_List_1.csv"
    
    master_df = pd.read_csv(master_csv_path)
    phot_df = pd.read_csv(master_phot_csv_path)
    index = master_df[master_df["hostname"] == host_name].index[0]
    phot_index = phot_df[phot_df["hostname"] == host_name].index[0]
    # Select row
    row = master_df.iloc[index]
    phot_row = phot_df.iloc[phot_index]

    # Build props dict
    props = {}

    # --- Parallax (required) ---
    # Need this for magnitudes to mean anything. 
    if not np.isnan(row["st_parallax_mas"]):
        props["parallax"] = (row["st_parallax_mas"], sigma_parallax)

    # band map from master_phot_csv to MIST keys
    band_map = {
        "gaiaGmag": "G_mag",
        "giaaBPmag": "BP_mag",
        "gaiaRPmag": "RP_mag",
        "Jmag": "J_mag",
        "Hmag": "H_mag",
        "Kmag": "K_mag",
    }
    #get photometry from phot_df
    for csv_col, iso_key in band_map.items():
        if csv_col in phot_row and not np.isnan(phot_row[csv_col]):
            sigmamag = 0.02 if sigma_mag is False else phot_row[f"e_{csv_col}"]
            props[iso_key] = phot_row[csv_col]
            props[f"{iso_key}_err"] = sigmamag


    return props

In [ ]:
# Load in the data using read_star_row_from_csv
sun = read_star_row_from_csv("HD 209458")
# Checking the data to make sure it was collected properly
print(sun)

{'parallax': (np.float64(20.7693898852824), 0.1), 'G_mag': np.float64(7.521244378), 'G_mag_err': np.float64(0.0183144879635296), 'RP_mag': np.float64(7.080287415), 'RP_mag_err': np.float64(0.0264984319384077), 'J_mag': np.float64(6.591), 'J_mag_err': np.float64(0.02), 'H_mag': np.float64(6.366), 'H_mag_err': np.float64(0.038), 'K_mag': np.float64(6.308), 'K_mag_err': np.float64(0.026)}


In [7]:
# Test minimize to give back how it is choosing.

